# Ring All-Reduce and DDP Gradient Averaging — Solution

**Difficulty**: 🔴 Hard

### Problem Statement

Every step of data-parallel training — PyTorch `DistributedDataParallel`, Horovod, the gradient
sync inside FSDP/ZeRO — ends with an **all-reduce**: each of the $W$ workers holds a local
gradient $g_r \in \mathbb{R}^N$ and every worker needs $\sum_{r} g_r$. For large messages NCCL's
classic algorithm is the **ring all-reduce** (Patarasuk & Yuan 2009; popularised for deep
learning by Baidu and Horovod in 2017). The ranks form a ring, every vector is cut into $W$
chunks, and in each of $2(W-1)$ steps every rank sends **one chunk to its right neighbour**:

1. **Reduce-scatter** ($W-1$ steps): partial sums travel around the ring, each hop adding the
   receiver's contribution. Afterwards each rank owns *one* chunk holding the complete sum.
2. **All-gather** ($W-1$ steps): the finished chunks travel around the ring and are *copied*,
   so every rank ends up with all of them.

Each rank sends $2\frac{W-1}{W}N$ elements — almost independent of $W$ — whereas in a naive
"gather to rank 0, sum, broadcast" scheme rank 0 alone must receive $(W-1)N$ and send $(W-1)N$.
Same total traffic, but the ring spreads it evenly over every link, which is why it scales.

You will simulate $W$ ranks inside one Python process. Ranks may only exchange data through
the provided `Network`, which records every message so the tests can verify the communication
pattern. Then you will use your all-reduce for the core of DDP: averaging gradients across
model replicas so that $W$ workers on $1/W$ of the batch each behave like one model on the
whole batch.

```
reduce-scatter, W = 3   (chunks a, b, c; subscripts = ranks whose data is summed in)

           rank 0           rank 1           rank 2
start      a0  b0  c0       a1  b1  c1       a2  b2  c2
step 0     sends a0 → 1     sends b1 → 2     sends c2 → 0
           c ← c0 + c2      a ← a1 + a0      b ← b2 + b1
step 1     sends c02 → 1    sends a01 → 2    sends b12 → 0
           b ← b0 + b12 ✔   c ← c1 + c02 ✔   a ← a2 + a01 ✔

rank r now owns the complete chunk (r + 1) % W. The all-gather passes the ✔ chunks
around the ring for W - 1 more steps, each receiver overwriting its stale copy.
```

---

### Requirements

1. **`ring_allreduce(tensors, net) -> list[Tensor]`**
   - `tensors`: a list of $W$ 1-D tensors of equal length $N$; `tensors[r]` is rank $r$'s local
     data. `net`: a `Network` with `net.world_size == W`.
   - Return a list of $W$ **new** tensors: `result[r]` is rank $r$'s copy of the elementwise sum
     of all inputs, with the inputs' shape and dtype. Do not modify the inputs.
   - Split each rank's working copy into $W$ contiguous chunks whose sizes differ by at most one
     (`torch.tensor_split` semantics — when $N < W$ some chunks are empty, which is fine). No
     padding.
   - **Reduce-scatter**, steps $s = 0, \dots, W-2$: every rank $r$ sends chunk $(r - s) \bmod W$
     to rank $(r+1) \bmod W$, which **adds** it into its own copy of that chunk. After the last
     step rank $r$ holds the fully reduced chunk $(r+1) \bmod W$.
   - **All-gather**, steps $s = 0, \dots, W-2$: every rank $r$ sends chunk $(r + 1 - s) \bmod W$
     to rank $(r+1) \bmod W$, which **overwrites** its copy of that chunk.
   - Communicate **only** through `net.send(src, dst, tensor)` and `net.recv(src, dst)`: rank $r$
     may use `tensors[r]` and whatever it receives — never `tensors[j]` for $j \ne r$. Rank $r$
     only ever sends to rank $(r+1) \bmod W$, and every message sent must be received.
   - Consequently each rank sends exactly $2(W-1)$ messages of $\lfloor N/W \rfloor$ or
     $\lceil N/W \rceil$ elements. With $W = 1$ nothing is sent and the result is a copy.
2. **`ddp_average_gradients(model_replicas, net) -> None`**
   - `model_replicas`: a list of $W$ structurally identical `nn.Module`s (replica $r$ lives on
     rank $r$), on which `backward()` has already run with each rank's own data shard.
   - Use the parameters with `requires_grad=True`, in `model.parameters()` order. Frozen
     parameters are skipped entirely (their `.grad` stays `None`). A trainable parameter whose
     `.grad` is `None` on some replica (it was unused in that replica's forward) contributes
     zeros.
   - Flatten each replica's gradients into **one** 1-D buffer (DDP's "bucket"), call
     `ring_allreduce` **once**, divide by $W$, and write each slice back as that parameter's
     `.grad` (same shape as the parameter) on every replica. Parameter values must not change.
   - Afterwards every replica holds the gradient one model would compute on the full batch,
     provided every rank used the same shard size and a mean-reduced loss.

---

### Constraints

- ✅ Use `torch.tensor_split`, `torch.cat`, `torch.split`, `view_as`, in-place ops on chunk views.
- ✅ Simulate the ranks with Python loops: within one step let every rank send, then every rank
  receive (that is what happens concurrently on real hardware).
- ❌ No `torch.distributed`.
- ❌ Never compute `sum(tensors)` or read another rank's tensor directly — the grader swaps in a
  network that delivers only zeros and checks that rank $r$'s output then depends on
  `tensors[r]` alone.
- ❌ No gather-to-one-rank and no passing whole vectors around the ring.

---

<details>
  <summary>💡 Hint</summary>

  - Each rank needs its own working buffer: `buf = tensors[r].clone()`, and `torch.tensor_split(buf, W)`
    gives $W$ *views* into it — an in-place `+=` or `copy_` on a view updates the buffer.
  - `torch.chunk` is not the same thing: it can return fewer than $W$ pieces when $N$ is not a
    multiple of $W$.
  - Track one chunk by hand for $W = 3$: who sends it at step 0, who adds to it at step 1, and
    who ends up holding the complete sum?
  - For DDP, the flat buffers must have the *same layout* on every replica, or the ring adds
    unrelated numbers together. What does that mean for a `None` gradient?

</details>

In [ ]:
import copy
from collections import defaultdict, deque

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class Network:
    """Simulated interconnect between `world_size` ranks that live in one process.

    net.send(src, dst, tensor)  rank `src` puts a COPY of `tensor` on the wire to rank `dst`
    net.recv(src, dst)          rank `dst` takes the oldest message from rank `src` (FIFO)
    net.log                     one (src, dst, numel) tuple per message, in send order
    net.pending()               number of messages sent but not yet received
    """

    def __init__(self, world_size):
        self.world_size = world_size
        self.log = []
        self._wire = defaultdict(deque)        # (src, dst) -> FIFO of tensors in flight

    def send(self, src, dst, tensor):
        W = self.world_size
        if not (0 <= src < W and 0 <= dst < W):
            raise ValueError(f"invalid ranks src={src}, dst={dst} for world_size={W}")
        if src == dst:
            raise ValueError(f"rank {src} tried to send a message to itself")
        self.log.append((src, dst, tensor.numel()))
        self._wire[(src, dst)].append(tensor.detach().clone())   # receiver gets its own copy

    def recv(self, src, dst):
        q = self._wire[(src, dst)]
        if not q:
            raise RuntimeError(f"rank {dst} is waiting for a message from rank {src} "
                               f"but none was sent (on a real cluster: deadlock)")
        return q.popleft()

    def pending(self):
        return sum(len(q) for q in self._wire.values())


def make_inputs(W, N, seed=0):
    """W ranks' local gradients: W random float64 vectors of length N."""
    g = torch.Generator().manual_seed(seed)
    return [torch.randn(N, generator=g, dtype=torch.float64) for _ in range(W)]


net = Network(2)
net.send(0, 1, torch.arange(3.0))
print("rank 1 received", net.recv(0, 1), "| log:", net.log, "| pending:", net.pending())

In [ ]:
def ring_allreduce(tensors, net):
    """Sum W equal-length 1-D tensors with a ring all-reduce through `net`.

    Args:
        tensors: list of W tensors of shape (N,); tensors[r] is rank r's local data.
        net: Network with net.world_size == W. The only way ranks may exchange data.

    Returns:
        list of W new tensors of shape (N,): result[r] is rank r's copy of the elementwise sum.
    """
    W = len(tensors)
    assert net.world_size == W, "one tensor per rank"
    # Each rank's private buffer, cut into W views (sizes differ by at most one, may be empty).
    bufs = [t.detach().clone() for t in tensors]
    chunks = [torch.tensor_split(b, W) for b in bufs]
    right = lambda r: (r + 1) % W
    left = lambda r: (r - 1) % W

    # Phase 1 — reduce-scatter: partial sums of chunk c start at rank c and move right,
    # each hop adding the receiver's contribution. After W-1 hops rank r owns chunk r+1.
    for s in range(W - 1):
        for r in range(W):                           # all ranks send ...
            net.send(r, right(r), chunks[r][(r - s) % W])
        for r in range(W):                           # ... then all ranks receive
            chunks[r][(r - 1 - s) % W].add_(net.recv(left(r), r))

    # Phase 2 — all-gather: the finished chunks move right and overwrite stale copies.
    for s in range(W - 1):
        for r in range(W):
            net.send(r, right(r), chunks[r][(r + 1 - s) % W])
        for r in range(W):
            chunks[r][(r - s) % W].copy_(net.recv(left(r), r))

    return bufs

In [ ]:
def ddp_average_gradients(model_replicas, net):
    """Average the gradients of W model replicas in place, like DDP after backward().

    Args:
        model_replicas: list of W identical nn.Modules; replica r has run backward() on its shard.
        net: Network with net.world_size == W.

    Returns:
        None. Every replica's trainable p.grad becomes the mean over replicas
        (None counts as zeros; frozen params are skipped).
    """
    W = len(model_replicas)
    params = [[p for p in m.parameters() if p.requires_grad] for m in model_replicas]
    if not params[0]:
        return
    # One flat "bucket" per replica, identical layout everywhere: a missing grad must still
    # occupy its slot (as zeros), or every later offset would shift on that replica.
    flat = [torch.cat([(p.grad if p.grad is not None else torch.zeros_like(p)).reshape(-1)
                       for p in ps])
            for ps in params]
    summed = ring_allreduce(flat, net)               # a single fused all-reduce
    sizes = [p.numel() for p in params[0]]
    for ps, buf in zip(params, summed):
        for p, g in zip(ps, torch.split(buf / W, sizes)):
            p.grad = g.view_as(p)                    # views into the bucket, like DDP's
                                                     # gradient_as_bucket_view=True

In [ ]:
# Test 1: every rank ends with the sum — including N % W != 0 and N < W
for W, N in [(1, 5), (2, 9), (3, 2), (4, 13), (4, 16), (7, 5), (7, 101)]:
    tensors = make_inputs(W, N, seed=W * 100 + N)
    originals = [t.clone() for t in tensors]
    out = ring_allreduce(tensors, Network(W))
    expected = torch.stack(tensors).sum(0)
    assert len(out) == W, f"expected {W} results, got {len(out)}"
    for r in range(W):
        assert out[r].shape == (N,) and out[r].dtype == torch.float64
        assert torch.allclose(out[r], expected, rtol=1e-12, atol=1e-12), f"W={W}, N={N}: rank {r} is wrong"
        assert torch.equal(out[r], out[0]), f"W={W}, N={N}: ranks 0 and {r} disagree bitwise"
        assert torch.equal(tensors[r], originals[r]), "the inputs must not be modified"
print("Test 1 passed: every rank holds the elementwise sum")

In [ ]:
# Test 2: the communication pattern is a ring — compare with "gather to rank 0, then broadcast"
W, N = 4, 1000
net = Network(W)
ring_allreduce(make_inputs(W, N), net)
assert all(dst == (src + 1) % W for src, dst, _ in net.log), "only send to the right neighbour"
assert net.pending() == 0, "every message sent must be received"
print(f"{'rank':>4} {'msgs sent':>10} {'elements sent':>14}   (ring, W={W}, N={N})")
for r in range(W):
    sent = [n for src, _, n in net.log if src == r]
    print(f"{r:>4} {len(sent):>10} {sum(sent):>14}")
    assert len(sent) == 2 * (W - 1), f"rank {r} should send 2(W-1) = {2 * (W - 1)} messages"
    assert sum(sent) == 2 * (W - 1) * N // W, f"rank {r} should send 2(W-1)N/W elements"
total = sum(n for _, _, n in net.log)
assert total == 2 * (W - 1) * N
print(f"busiest rank sends {2 * (W - 1) * N // W} elements = 2(W-1)/W·N "
      f"(naive scheme: rank 0 receives {(W - 1) * N} and sends {(W - 1) * N})")
print("Test 2 passed: ring traffic pattern")

In [ ]:
# Test 3: DDP — W replicas on 1/W of the batch each == one model on the whole batch
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(5, 8)
        self.fc2 = nn.Linear(8, 3, bias=False)
        self.temperature = nn.Parameter(torch.tensor(1.5))                 # 0-dim parameter
        self.offset = nn.Parameter(torch.zeros(3), requires_grad=False)    # frozen

    def forward(self, x):
        return self.fc2(torch.tanh(self.fc1(x))) / self.temperature + self.offset

torch.manual_seed(0)
W, per_rank = 4, 3
base = TinyNet().double()
X = torch.randn(W * per_rank, 5, dtype=torch.float64)
Y = torch.randn(W * per_rank, 3, dtype=torch.float64)

single = copy.deepcopy(base)
F.mse_loss(single(X), Y).backward()                    # full batch, mean loss

replicas = [copy.deepcopy(base) for _ in range(W)]
for model, xs, ys in zip(replicas, X.chunk(W), Y.chunk(W)):
    F.mse_loss(model(xs), ys).backward()               # each rank: its own shard, mean loss

net = Network(W)
ddp_average_gradients(replicas, net)
for r, model in enumerate(replicas):
    for (name, p_ref), p in zip(single.named_parameters(), model.parameters()):
        if p_ref.grad is None:
            assert p.grad is None, f"frozen '{name}' must keep grad=None"
        else:
            assert torch.allclose(p.grad, p_ref.grad, rtol=1e-10, atol=1e-12), \
                f"replica {r}: '{name}'.grad differs from full-batch training"
msgs = [sum(1 for src, _, _ in net.log if src == r) for r in range(W)]
assert msgs == [2 * (W - 1)] * W, f"expected ONE fused all-reduce (2(W-1) messages per rank), got {msgs}"
print("Test 3 passed: averaged replica gradients == full-batch gradients")

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("ring-allreduce", ring_allreduce=ring_allreduce, ddp_average_gradients=ddp_average_gradients)

### Why it works

**Reduce-scatter.** Chunk $c$ is first sent by rank $c$ (step 0), and at step $s$ it is sent by
rank $c+s$ and added into rank $c+s+1$'s copy — so after step $s$ rank $c+s+1$ holds the sum
of $s+2$ ranks' contributions $c, c+1, \dots, c+s+1$. After the last step ($s = W-2$) rank
$c + W - 1 \equiv c - 1$ holds all $W$ contributions: **rank $r$ owns chunk $r+1$**. Different
ranks always send different chunks within a step, and the chunk a rank sends is never the one
it receives in that step, so there are no read/write hazards.

**All-gather.** At step 0 each rank sends the chunk it owns; afterwards it forwards the chunk it
just received. Chunk $c$ visits ranks $c, c+1, \dots, c+W-2$ — everyone except its owner $c-1$
— and is *copied*, never re-added. Because every rank copies the very same finished bits, all
ranks end bitwise identical: DDP replicas never drift apart.

**Cost.** Each rank sends $2(W-1)$ messages of $N/W$ elements. With the usual
$\alpha$ (per-message latency) / $\beta$ (seconds per element) model,

$$T_{\text{ring}} = 2(W-1)\,\alpha + 2\,\frac{W-1}{W}\,N\beta ,$$

and $2\frac{W-1}{W}N$ is the lower bound on the data any rank must send for an all-reduce, so
the ring is bandwidth-optimal; its weakness is the latency term, linear in $W$.

**DDP.** With a mean loss over a shard of $n = B/W$ samples,
$\frac1W\sum_r \nabla \frac1n\sum_{i\in r}\ell_i = \frac1{Wn}\sum_i \nabla\ell_i = \nabla\frac1B\sum_i\ell_i$:
averaging per-replica gradients equals the full-batch gradient exactly. Flattening all
gradients into one buffer turns many small all-reduces (each paying $2(W-1)\alpha$) into one.

### Complexity

- Per rank: $2(W-1)$ messages, $2\frac{W-1}{W}N$ elements sent and received, $\frac{W-1}{W}N$ additions.
- Memory: one working copy of $N$ per rank (plus one chunk in flight).
- `ddp_average_gradients`: one all-reduce of $P$ (trainable parameter count) elements plus $O(P)$ copies.

### Common bugs

- `torch.chunk` instead of `torch.tensor_split`: `chunk` returns fewer than $W$ pieces when $N$ is
  not a multiple of $W$ (e.g. $N=6, W=4$ gives 3 chunks), so indexing chunk 3 crashes or the
  schedule silently skips data.
- `+=` in the all-gather phase instead of a copy — finished chunks get summed again.
- Off-by-one chunk indices: the receiver at step $s$ updates chunk $(r-1-s)$, the chunk its
  *left* neighbour sent, not $(r-s)$.
- Interleaving send and receive per rank without double-buffering: on real hardware the send
  of step $s$ must not see data that arrived in step $s$ (here the network copies messages and
  the chunk indices never collide, but a buffer reused across steps can).
- Mutating `tensors[r]` in place — the caller (DDP's autograd hooks, the optimizer) still owns it.
- DDP: forgetting to divide by $W$ (that is a $W\times$ larger learning rate); one all-reduce
  per parameter (correct numbers, latency-bound: hundreds of tiny messages); dropping `None`
  gradients from the flat buffer — the layouts then differ between replicas and the ring adds
  unrelated numbers; giving frozen parameters a zero `.grad` (an optimizer with weight decay
  would then update them).

### Interview follow-ups

1. **Why is the ring bandwidth-optimal, and what is the lower bound?** Every rank must ship out
   at least $\frac{W-1}{W}N$ of its data to be reduced elsewhere and receive $\frac{W-1}{W}N$ of
   finished results, so $2\frac{W-1}{W}N$ per rank is a lower bound — the ring achieves it, and
   it tends to $2N$ independent of $W$. The naive gather-to-root sends the same total, but all
   through one rank's links: $(W-1)N$ in and $(W-1)N$ out.
2. **What about latency, and when does a tree win?** The ring takes $2(W-1)$ sequential steps,
   so for small messages or thousands of GPUs the $2(W-1)\alpha$ term dominates. Tree-based
   all-reduce (NCCL's double binary trees) needs $O(\log W)$ steps with near-full bandwidth;
   NCCL picks ring vs tree by message size and topology, and runs hierarchically (NVLink inside
   a node, InfiniBand between nodes).
3. **How does PyTorch DDP make this fast in practice?** It registers autograd hooks, groups
   parameters into ~25 MB buckets (in roughly reverse order, since gradients become ready from
   the last layer backwards) and launches an asynchronous all-reduce as soon as a bucket is
   full — overlapping communication with the rest of backward. Bucketing is exactly this
   problem's flattening: it amortizes the per-message latency. It pre-divides by the world size
   and reduces with SUM.
4. **Why do unequal shard sizes break the average?** $\frac1W\sum_r \frac1{n_r}\sum_{i\in r}\nabla\ell_i$
   weights a sample by $\frac{1}{W n_r}$ instead of $\frac1B$, over-weighting ranks with smaller
   shards. Fixes: equal shards (`DistributedSampler` pads/drops), or sum the losses and divide
   by the global batch size obtained with an all-reduce of the counts.
5. **How do reduce-scatter and all-gather relate to FSDP / ZeRO?** All-reduce = reduce-scatter
   + all-gather. ZeRO-2/FSDP stop after the reduce-scatter — each rank keeps only its shard of
   the gradient and updates only its shard of the optimizer state — and FSDP/ZeRO-3 all-gather
   the parameters right before each layer's forward and backward. Same per-rank bandwidth per
   collective as DDP, ~1.5× DDP's traffic overall for ZeRO-3, in exchange for $W\times$ less memory.
6. **Are the results deterministic?** Each chunk is summed in a different ring order, so results
   differ from a sequential sum in the last bits (more so in bf16), but every rank receives the
   same bits. Changing $W$ or the algorithm (ring vs tree) changes the rounding — one reason
   runs are not bitwise reproducible across cluster sizes.